# async

In [ ]:
///- --test static

open testing

In [ ]:
open rust
open rust_operators

### base_let'

In [ ]:
inl base_let' x =
    let' x

## rust

### future

In [ ]:
nominal future t =
    `(
        rust_type
            `($'std_future_Future<`t>')
            `($'std::future::Future<Output = `t>')
            "type std_future_Future<'T> = class end"
    )

### future_pin

In [ ]:
type future_pin t = rust.pin (rust.box (rust.dyn' (future t)))

### future_pin_send

In [ ]:
type future_pin_send t = rust.pin (rust.box (rust.send (rust.dyn' (future t))))

### block_on_tokio

In [ ]:
inl block_on_tokio forall t. (fn : future_pin t) : t =
    inl runtime : infer =
        !\($'$"tokio::runtime::Builder::new_multi_thread().enable_all().build().unwrap()"')
    !\\(fn, $'"!runtime.handle().block_on($0)"')

### block_on_futures_lite

In [ ]:
inl block_on_futures_lite forall t. (fn : future_pin t) : t =
    !\\(fn, $'"futures_lite::future::block_on($0)"')

### block_on_futures

In [ ]:
inl block_on_futures forall t. (fn : future_pin t) : t =
    !\\(fn, $'"futures::executor::block_on($0)"')

### block_on_async_std

In [ ]:
inl block_on_async_std forall t. (fn : future_pin t) : t =
    !\\(fn, $'"async_std::task::block_on($0)"')

### block_on_tokio_send

In [ ]:
inl block_on_tokio_send forall t. (fn : future_pin_send t) : t =
    !\($'"tokio::runtime::block_on(!fn)"')

### stream_ext_tokio

In [ ]:
nominal stream_ext_tokio =
    `(
        rust_type
            `($'tokio_stream_StreamExt')
            `($'tokio_stream::StreamExt')
            "type tokio_stream_StreamExt = class end"
    )

### join_handle_tokio

In [ ]:
nominal join_handle_tokio t =
    `(
        rust_type
            `($'tokio_task_JoinHandle<`t>')
            `($'tokio::task::JoinHandle<`t>')
            "type tokio_task_JoinHandle<'T> = class end"
    )

### stream_collect_tokio

In [ ]:
inl stream_collect_tokio forall t u.
    (stream : t)
    : future_pin (am'.vec u)
    =
    !\($'"Box::pin(tokio_stream::StreamExt::collect(!stream))"')

### stream_collect_futures

In [ ]:
inl stream_collect_futures forall t u.
    (stream : t)
    : future_pin (am'.vec u)
    =
    !\($'"Box::pin(futures::stream::StreamExt::collect(!stream))"')

### stream_next_tokio

In [ ]:
inl stream_next_tokio forall t u.
    (stream : t)
    : future_pin (optionm'.option' u)
    =
    !\($'"let mut !stream = !stream"')
    !\($'"Box::pin(tokio_stream::StreamExt::next(&mut !stream))"')

### stream_filter_map_tokio

In [ ]:
inl stream_filter_map_tokio forall t u v.
    (fn : u -> optionm'.option' v)
    (stream : t)
    : infer' v
    =
    inl fn = join fn
    !\($'"tokio_stream::StreamExt::filter_map(!stream, |x| !fn(x))"')

### stream_filter_map_futures

In [ ]:
inl stream_filter_map_futures forall t u v.
    (fn : u -> optionm'.option' v)
    (stream : t)
    : infer' v
    =
    inl fn = join fn
    !\($'"futures::stream::StreamExt::filter_map(!stream, |x| async { !fn(x) })"')

### spawn_tokio

In [ ]:
inl spawn_tokio forall t. (fn : future_pin_send t) : join_handle_tokio t =
    !\($'"tokio::runtime::spawn(!fn)"')

### try_join_all

In [ ]:
nominal try_join_all t =
    `(
        rust_type
            `($'futures_future_TryJoinAll<`t>')
            `($'futures::future::TryJoinAll<`t>')
            "type futures_future_TryJoinAll<'T> = class end"
    )

inl try_join_all forall t. (x : am'.vec (future_pin (resultm.result' t sm'.std_string))) : try_join_all (future_pin (resultm.result' t sm'.std_string)) =
    inl x = join x
    !\($'"futures::future::try_join_all(!x)"')

### fuse_tokio

In [ ]:
nominal fuse_tokio t =
    `(
        rust_type
            `($'tokio_prelude_stream_Fuse<`t>')
            `($'tokio::prelude::stream::Fuse<`t>')
            "type tokio_prelude_stream_Fuse<'T> = class end"
    )

### fuse'

In [ ]:
type fuse' t = fuse_tokio t

### future_fuse

In [ ]:
inl future_fuse forall t. (x : future_pin t) : fuse' (future_pin t) =
    !\($'"futures::future::FutureExt::fuse(!x)"')

### join_all

In [ ]:
nominal join_all t =
    `(
        rust_type
            `($'futures_future_JoinAll<`t>')
            `($'futures::future::JoinAll<`t>')
            "type futures_future_JoinAll<'T> = class end"
    )

inl join_all forall t. (x : am'.vec (future_pin t)) : join_all (future_pin t) =
    inl x = join x
    !\($'"futures::future::join_all(!x)"')

### join_all_send

In [ ]:
inl join_all_send forall t. (x : am'.vec (future_pin_send t)) : join_all (future_pin_send t) =
    inl x = join x
    !\($'"futures::future::join_all(!x)"')

### join_handle'

In [ ]:
type join_handle' t = join_handle_tokio t

### await_handle

In [ ]:
inl await_handle forall t. (x : join_handle' t) : t =
    !\($'"!x.await"')

### await_all

In [ ]:
inl await_all forall t. (x : join_all (future_pin t)) : am'.vec t =
    !\($'"!x.await"')

### await_all_send

In [ ]:
inl await_all_send forall t. (x : join_all (future_pin_send t)) : am'.vec t =
    !\($'"!x.await"')

### try_await_all

In [ ]:
inl try_await_all forall t. (x : try_join_all (future_pin (resultm.result' t sm'.std_string))) : resultm.result' (am'.vec t) sm'.std_string =
    !\($'"!x.await"')

### try_await_all_send

In [ ]:
inl try_await_all_send forall t. (x : try_join_all (future_pin_send (resultm.result' t sm'.std_string))) : resultm.result' (am'.vec t) sm'.std_string =
    !\($'"!x.await"')

### await

In [ ]:
inl await forall t. (x : future_pin t) : t =
    !\($'"!x.await"')

### await

In [ ]:
inl await_send forall t. (x : future_pin_send t) : t =
    !\($'"!x.await"')

### into_iter

In [ ]:
type into_iter t = array_base t

### into_par_iter

In [ ]:
inl into_par_iter forall t. (x : am'.vec t) : into_iter t =
    x |> am'.from_vec_base

### par_map

In [ ]:
inl par_map forall t u. (fn : t -> u) (ar : into_iter t) : into_iter u =
    inl n : i32 = $'(!ar.borrow().len() as i32)'
    let rec loop (i : i32) (acc : am'.vec u) : am'.vec u =
        if i < n then loop (i + 1) (acc |> am'.vec_push (fn (am'.index_base i ar))) else acc
    loop 0 (am'.new_vec []) |> am'.from_vec_base

### par_collect

In [ ]:
inl par_collect forall t. (iter : into_iter t) : am'.vec t =
    iter |> am'.to_vec

### try_join_all_iter

In [ ]:
inl try_join_all_iter forall t. (x : am'.vec (future_pin_send (resultm.result' t sm'.std_string))) : try_join_all (future_pin_send (resultm.result' t sm'.std_string)) =
    inl x = join x
    !\($'"futures::future::try_join_all(!x)"')

### future_init

In [ ]:
inl future_init forall t. (move : bool) (x : () -> t) : infer' t =
    (!\($'"true; let __future_init = Box::pin(/*"') : bool) |> ignore
    if move
    then (!\($'"*/ async move { /*"') : bool) |> ignore
    else (!\($'"*/ async { /*"') : bool) |> ignore
    (!\($'"*/ ()"') : bool) |> ignore

    inl x' = x ()

    inl depth = 1, 0

    x' |> rust.fix_closure depth

    !\($'"__future_init"')

### new_future

In [ ]:
inl new_future forall t. (x : () -> t) : future_pin t =
    inl result = future_init false x
    !\($'"!result"')

### new_future_move

In [ ]:
inl new_future_move forall t. (x : () -> t) : future_pin t =
    inl result = future_init true x
    !\($'"!result"')

### new_future_send

In [ ]:
inl new_future_send forall t. (x : () -> t) : future_pin_send t =
    inl result = future_init false x
    !\($'"!result"')

### new_future_move_send

In [ ]:
inl new_future_move_send forall t. (x : () -> t) : future_pin_send t =
    inl result = future_init true x
    !\($'"!result"')

## fsharp

### async

In [ ]:
nominal async_python t =
    `(
        backend_switch `(()) `({}) {
            Python = (fun () => global "import asyncio") : () -> ()
        }
        $'' : $'any'
    )
type async_switch t =
    {
        Fsharp : $'Async<`t>'
        Python : async_python t
        Rust : $'std::pin::Pin<Box<dyn std::future::Future<Output = `t>>>'
    }
nominal async t = $'backend_switch `(async_switch t)'

### task

In [ ]:
nominal task t =
    `(
        typecase t with
        | () => $'' : $'System.Threading.Tasks.Task'
        | _ => $'' : $'System.Threading.Tasks.Task<`t>'
    )

### new_task

In [ ]:
inl new_task forall t. (fn : () -> t) : task t =
    run_target_args' fn function
        | Fsharp _ => fun fn =>
            inl result : optionm'.option' (task t) = optionm'.none' ()
            $'let mutable _new_task_!result = !result '
            $'task {'
            fn () |> ignore
            $'}'
            $'|> fun x -> _new_task_!result <- Some x'
            $'match _new_task_!result with Some x -> x | None -> failwith "async.new_task / _new_task_!result=None"'
        | _ => fun _ => null ()

### await_task

In [ ]:
inl await_task forall t. (a : task t) : async t =
    run_target function
        | Fsharp _
        | TypeScript _ => fun () =>
            a |> $'Async.AwaitTask'
        | _ => fun () => null ()

### ignore

In [ ]:
inl ignore forall t. (a : async t) : async () =
    if is_rust ()
    then $'Box::pin(async move { let _ = !a.await; })' : async ()
    else
        run_target function
            | Fsharp _
            | TypeScript _ => fun () =>
                a |> $'Async.Ignore'
            | _ => fun () => null ()

### run_synchronously

In [ ]:
inl run_synchronously forall t. (a : async t) : t =
    run_target function
        | Fsharp _ => fun () =>
            a |> $'Async.RunSynchronously'
        | Cuda => fun () =>
            $'asyncio.run(!a())'
        | Rust => fun () =>
            $'{ struct SpiralThreadWaker(std::thread::Thread); impl std::task::Wake for SpiralThreadWaker { fn wake(self: std::sync::Arc<Self>) { self.0.unpark(); } } let waker = std::task::Waker::from(std::sync::Arc::new(SpiralThreadWaker(std::thread::current()))); let mut cx = std::task::Context::from_waker(&waker); let mut f = !a; loop { match std::future::Future::poll(f.as_mut(), &mut cx) { std::task::Poll::Ready(v) => break v, std::task::Poll::Pending => std::thread::park() } } }' : t
        | _ => fun () => null ()

inl catch_rust forall t. (a : async t) : async (resultm.result' t exn) =
    $'Box::pin({ thread_local\! { static QUIET: std::cell::Cell<u32> = std::cell::Cell::new(0); } static HOOK: std::sync::Once = std::sync::Once::new(); HOOK.call_once(|| { let prev = std::panic::take_hook(); std::panic::set_hook(Box::new(move |info| if cfg\!(panic = "abort") || QUIET.with(|q| q.get()) == 0 { prev(info) })) }); let mut f = !a; std::future::poll_fn(move |cx| { QUIET.with(|q| q.set(q.get() + 1)); let r = std::panic::catch_unwind(std::panic::AssertUnwindSafe(|| std::future::Future::poll(f.as_mut(), cx))); QUIET.with(|q| q.set(q.get() - 1)); match r { Ok(std::task::Poll::Ready(v)) => std::task::Poll::Ready(Ok(v)), Ok(std::task::Poll::Pending) => std::task::Poll::Pending, Err(e) => std::task::Poll::Ready(Err(Rc::<str>::from(e.downcast_ref::<String>().map(|s| s.as_str()).or_else(|| e.downcast_ref::<&str>().copied()).unwrap_or("panic")))) } }) })' : async (resultm.result' t exn)

inl start_child_rust forall t. (timeout : i32) (a : async t) : async (async t) =
    $'Box::pin({ thread_local\! { static QUIET: std::cell::Cell<u32> = std::cell::Cell::new(0); } static HOOK: std::sync::Once = std::sync::Once::new(); HOOK.call_once(|| { let prev = std::panic::take_hook(); std::panic::set_hook(Box::new(move |info| if cfg\!(panic = "abort") || QUIET.with(|q| q.get()) == 0 { prev(info) })) }); let mut f = !a; let timeout_ms: i32 = !timeout; async move { let deadline = if timeout_ms >= 0 { Some(std::time::Instant::now() + std::time::Duration::from_millis(timeout_ms as u64)) } else { None }; let first = std::future::poll_fn(|cx| { QUIET.with(|q| q.set(q.get() + 1)); let r = std::panic::catch_unwind(std::panic::AssertUnwindSafe(|| std::future::Future::poll(f.as_mut(), cx))); QUIET.with(|q| q.set(q.get() - 1)); std::task::Poll::Ready(r) }).await; let mut done = match first { Ok(std::task::Poll::Ready(v)) => Some(Ok((v, std::time::Instant::now()))), Ok(std::task::Poll::Pending) => None, Err(e) => Some(Err(e)) }; let mut timer: Option<std::sync::Arc<std::sync::Mutex<std::task::Waker>>> = None; let h: std::pin::Pin<Box<dyn std::future::Future<Output = `t>>> = Box::pin(std::future::poll_fn(move |cx| { if done.is_none() { if let std::task::Poll::Ready(v) = std::future::Future::poll(f.as_mut(), cx) { done = Some(Ok((v, std::time::Instant::now()))) } } match done.take() { Some(Err(e)) => std::panic::resume_unwind(e), Some(Ok((v, at))) => if deadline.map_or(false, |d| at > d) { panic\!("The operation has timed out.") } else { std::task::Poll::Ready(v) }, None => match deadline { Some(d) if std::time::Instant::now() >= d => panic\!("The operation has timed out."), Some(d) => { match &timer { Some(w) => *w.lock().unwrap() = cx.waker().clone(), None => { let w = std::sync::Arc::new(std::sync::Mutex::new(cx.waker().clone())); let w2 = w.clone(); std::thread::spawn(move || { let now = std::time::Instant::now(); if d > now { std::thread::sleep(d - now) } w2.lock().unwrap().wake_by_ref() }); timer = Some(w) } } std::task::Poll::Pending }, None => std::task::Poll::Pending } } })); h } })' : async (async t)

### start

In [ ]:
inl start (a : async ()) : () =
    if is_rust ()
    then a |> run_synchronously
    else
        run_target function
            | Fsharp _
            | TypeScript _ => fun () =>
                a |> $'Async.Start'
            | _ => fun () => null ()

### start_child

In [ ]:
inl start_child forall t. (a : async t) : async (async t) =
    if is_rust ()
    then a |> start_child_rust -1
    else
        run_target function
            | Fsharp _
            | TypeScript _ => fun () =>
                a |> $'Async.StartChild'
            | _ => fun () => null ()

### start_child_timeout

In [ ]:
inl start_child_timeout forall t. (timeout : i32) (a : async t) : async (async t) =
    if is_rust ()
    then a |> start_child_rust timeout
    else
        run_target function
            | Fsharp _
            | TypeScript _ => fun () =>
                $'Async.StartChild (!a, !timeout)'
            | _ => fun () => null ()

### start_immediate

In [ ]:
inl start_immediate forall t. (a : async t) : () =
    if is_rust ()
    then a |> run_synchronously |> fun _ => ()
    else
        run_target function
            | Fsharp _
            | TypeScript _ => fun () =>
                a |> $'Async.StartImmediate'
            | _ => fun () => null ()

### start_with_continuations

In [ ]:
inl start_with_continuations forall t. (a : async t) : () =
    if is_rust ()
    then a |> catch_rust |> run_synchronously |> fun _ => ()
    else
        run_target_args' a function
            | Fsharp _
            | TypeScript _ => fun a =>
                $'Async.StartWithContinuations (!a, ignore, ignore, ignore)'
            | _ => fun _ => null ()

### task_canceled_exception

In [ ]:
nominal task_canceled_exception = $'System.Threading.Tasks.TaskCanceledException'

### sleep

In [ ]:
inl sleep (ms : i32) : async () =
    run_target function
        | Fsharp _
        | TypeScript _ => fun () =>
            ms |> $'Async.Sleep'
        | Cuda _ => fun () =>
            $'asyncio.sleep(!ms / 1000)'
        | Rust => fun () =>
            $'Box::pin(async move { std::thread::sleep(std::time::Duration::from_millis((!ms as u64))) })' : async ()
        | _ => fun () => null ()

### do

In [ ]:
inl do (a : async ()) : () =
    backend_switch {
        Fsharp = fun () => $'do\! !a ' : ()
        Python = fun () => $'await !a ' : ()
    }

### let'

In [ ]:
inl let' forall t. (a : async t) : t =
    $'let\! !a = !a '
    $'!a '

### return_await

In [ ]:
inl return_await forall t. (a : async t) : () =
    backend_switch {
        Fsharp = fun () => $'return\! !a ' : ()
        Python = fun () => $'asyncio.run(!a())' : ()
    }

### return_await'

In [ ]:
inl return_await' forall t. (a : async t) : t =
    backend_switch {
        Fsharp = fun () => $'return\! !a ' : ()
        Python = fun () => $'await !a()' : ()
    }

### catch'

In [ ]:
inl catch' forall t e. (a : async t) : async (choice2' t e) =
    if is_rust ()
    then
        inl r = a |> catch_rust
        $'(!r)' : async (choice2' t e)
    else
        run_target function
            | Fsharp _
            | TypeScript _ => fun () =>
                a |> $'Async.Catch'
            | _ => fun () => null ()

### cancellation_token

In [ ]:
inl cancellation_token () : async threading.cancellation_token =
    $'Async.CancellationToken'

inl default_cancellation_token () : threading.cancellation_token =
    $'Async.DefaultCancellationToken'

### value_task

In [ ]:
nominal value_task = $'System.Threading.Tasks.ValueTask'

### value_task_as_task

In [ ]:
inl value_task_as_task (task : value_task) : task () =
    run_target function
        | Fsharp => fun () => $'!task.AsTask' ()
        | _ => fun () => null ()

### await_value_task_unit

In [ ]:
inl await_value_task_unit (task : value_task) : async () =
    task |> value_task_as_task |> await_task

## main

In [1]:
inl main () =
    ()